<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/0811_Hugging_Face_%EC%B0%A8%EC%84%A0_%EA%B2%80%EC%B6%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch torchvision opencv-python pillow

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

def process_video_all_objects(input_video, output_video):
    """모든 객체를 색상별로 표시하는 세그멘테이션"""

    print(f"🎬 {input_video} 처리 시작...")

    # 1. Hugging Face 세그멘테이션 모델 로드
    try:
        segmenter = pipeline(
            "image-segmentation",
            model="nvidia/segformer-b0-finetuned-cityscapes-1024-1024",
            device=0 if torch.cuda.is_available() else -1
        )
        print("✅ 모델 로드 완료")
    except Exception as e:
        print(f"❌ 모델 로드 실패: {e}")
        return

    # 2. 객체별 색상 정의 (BGR 형식)
    colors = {
        'road': [0, 255, 0],          # 녹색
        'sidewalk': [255, 255, 0],    # 노란색
        'building': [128, 128, 128],   # 회색
        'wall': [128, 0, 0],          # 갈색
        'fence': [255, 0, 255],       # 마젠타
        'pole': [0, 255, 255],        # 시안
        'traffic light': [0, 0, 255], # 빨간색 - 신호등
        'traffic sign': [255, 255, 255], # 흰색 - 표지판
        'vegetation': [0, 128, 0],     # 어두운 녹색
        'terrain': [128, 64, 0],      # 갈색
        'sky': [255, 128, 0],         # 하늘색
        'person': [255, 0, 0],        # 파란색 - 사람
        'rider': [128, 0, 255],       # 보라색
        'car': [0, 0, 128],           # 어두운 빨간색 - 자동차
        'truck': [255, 0, 128],       # 핑크 - 트럭
        'bus': [128, 255, 0],         # 연두색 - 버스
        'train': [0, 128, 255],       # 주황색
        'motorcycle': [255, 128, 128], # 연분홍 - 오토바이
        'bicycle': [128, 255, 255]     # 연청색 - 자전거
    }

    # 3. 비디오 열기
    cap = cv2.VideoCapture(input_video)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"📹 비디오: {width}x{height}, {fps}fps, {total_frames}프레임")

    # 4. 출력 비디오 설정
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        try:
            # RGB 변환
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)

            # 세그멘테이션 실행
            results = segmenter(pil_image)

            # 오버레이 이미지 생성
            overlay = np.zeros_like(frame)
            detected_objects = []

            # 모든 세그멘테이션 결과 처리
            for result in results:
                label = result['label'].lower()
                mask = np.array(result['mask'])

                # 해당 객체의 색상 가져오기
                if label in colors:
                    color = colors[label]
                    overlay[mask] = color
                    detected_objects.append(label)
                else:
                    # 정의되지 않은 객체는 기본 색상
                    overlay[mask] = [64, 64, 64]  # 어두운 회색
                    detected_objects.append(label)

            # 원본 프레임과 오버레이 합성
            result_frame = cv2.addWeighted(frame, 0.6, overlay, 0.4, 0)

            # 화면에 검출된 객체 목록 표시
            unique_objects = list(set(detected_objects))
            y_offset = 30

            # 배경 박스 그리기 (텍스트 가독성)
            cv2.rectangle(result_frame, (10, 10), (400, 30 + len(unique_objects) * 25), (0, 0, 0), -1)

            for i, obj in enumerate(unique_objects):
                if obj in colors:
                    color = colors[obj]
                    # 객체명과 색상 표시
                    cv2.putText(result_frame, f"{obj}", (15, y_offset + i*25),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # 프레임 정보 표시
            cv2.putText(result_frame, f"Frame: {frame_count}/{total_frames}",
                       (width-200, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            out.write(result_frame)

        except Exception as e:
            print(f"프레임 {frame_count} 처리 실패: {e}")
            # 실패 시 원본 프레임 사용
            out.write(frame)

        frame_count += 1

        # 진행률 출력
        if frame_count % 30 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"⏳ 진행률: {progress:.1f}% | 검출된 객체: {len(unique_objects)}개")

    cap.release()
    out.release()

    print(f"✅ 완료! 결과: {output_video}")
    print(f"총 {frame_count}프레임 처리됨")

# 디버그용: 한 프레임만 테스트
def test_single_frame(video_path):
    """한 프레임만 테스트해서 뭐가 검출되는지 확인"""

    segmenter = pipeline(
        "image-segmentation",
        model="nvidia/segformer-b0-finetuned-cityscapes-1024-1024",
        device=0 if torch.cuda.is_available() else -1
    )

    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()

    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb)

        results = segmenter(pil_image)

        print("🔍 검출된 객체들:")
        for i, result in enumerate(results):
            print(f"{i+1}. {result['label']}")

        return results
    else:
        print("❌ 프레임 읽기 실패")
        return None

# 실행 옵션들
print("=== 세그멘테이션 테스트 ===")
print("1. 한 프레임 테스트:")
print("test_single_frame('/content/2.mp4')")
print("\n2. 전체 비디오 처리:")
print("#process_video_all_objects('/content/2.mp4', '/content/2_all_objects.mp4')")

# 실행
#test_single_frame('/content/1.mp4')  # 먼저 테스트
process_video_all_objects('/content/2.mp4', '/content/2_all_objects.mp4')  # 전체 처리

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image
from transformers import pipeline
import warnings
import time
from IPython.display import HTML, display
from base64 import b64encode

warnings.filterwarnings('ignore')

def process_video_all_objects(input_video, output_video):
    """모든 객체를 색상별로 표시하고, 실시간 진행률과 ETA를 보여주며, 최종 영상을 Colab에 출력합니다."""

    print(f"🎬 '{input_video}' 파일 처리를 시작합니다...")

    # 1. Hugging Face 세그멘테이션 모델 로드
    try:
        device = 0 if torch.cuda.is_available() else -1
        if device == 0:
            print(f"✅ CUDA GPU를 사용하여 모델을 로드합니다. (device: {torch.cuda.get_device_name(0)})")
        else:
            print("⚠️ CUDA GPU를 찾을 수 없습니다. CPU로 모델을 로드합니다. 처리 속도가 매우 느릴 수 있습니다.")

        segmenter = pipeline(
            "image-segmentation",
            model="nvidia/segformer-b0-finetuned-cityscapes-1024-1024",
            device=device
        )
        print("✅ 모델 로드 완료")
    except Exception as e:
        print(f"❌ 모델 로드에 실패했습니다: {e}")
        return

    # 2. 객체별 색상 정의 (BGR 형식)
    colors = {
        'road': [0, 255, 0], 'sidewalk': [255, 255, 0], 'building': [128, 128, 128],
        'wall': [128, 0, 0], 'fence': [255, 0, 255], 'pole': [0, 255, 255],
        'traffic light': [0, 0, 255], 'traffic sign': [255, 255, 255],
        'vegetation': [0, 128, 0], 'terrain': [128, 64, 0], 'sky': [255, 128, 0],
        'person': [255, 0, 0], 'rider': [128, 0, 255], 'car': [0, 0, 128],
        'truck': [255, 0, 128], 'bus': [128, 255, 0], 'train': [0, 128, 255],
        'motorcycle': [255, 128, 128], 'bicycle': [128, 255, 255]
    }

    # 3. 비디오 열기
    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        print(f"❌ 비디오 파일을 열 수 없습니다: {input_video}")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"📹 비디오 정보: {width}x{height}, {fps}fps, 총 {total_frames}프레임")

    # 4. 출력 비디오 설정
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    frame_count = 0
    start_time = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        frame_start_time = time.time()

        try:
            pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            results = segmenter(pil_image)

            overlay = np.zeros_like(frame)
            detected_objects = set()

            for result in results:
                label = result['label'].lower()
                mask = np.array(result['mask'])
                detected_objects.add(label)
                color = colors.get(label, [64, 64, 64]) # 정의되지 않은 객체는 어두운 회색
                overlay[mask] = color

            result_frame = cv2.addWeighted(frame, 0.6, overlay, 0.4, 0)

            # --- 화면에 정보 표시 ---
            y_offset = 30
            # 텍스트 가독성을 위한 배경 박스
            cv2.rectangle(result_frame, (5, 5), (300, 25 + len(detected_objects) * 20), (0, 0, 0), -1)

            # 검출된 객체 목록
            for i, obj in enumerate(sorted(list(detected_objects))):
                color = colors.get(obj, (200, 200, 200))
                cv2.putText(result_frame, obj, (10, y_offset + i*20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

            # 프레임 정보
            cv2.putText(result_frame, f"Frame: {frame_count}/{total_frames}", (width-150, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            out.write(result_frame)

        except Exception as e:
            print(f"⚠️ 프레임 {frame_count} 처리 중 오류 발생: {e}. 원본 프레임을 저장합니다.")
            out.write(frame)

        # --- 실시간 진행률 및 ETA 출력 (매 프레임마다) ---
        frame_time = time.time() - frame_start_time
        elapsed_time = time.time() - start_time
        avg_time_per_frame = elapsed_time / frame_count
        remaining_frames = total_frames - frame_count
        eta_seconds = remaining_frames * avg_time_per_frame

        eta_str = time.strftime('%M분 %S초', time.gmtime(eta_seconds))

        # <<<✨✨✨ 수정된 부분 ✨✨✨>>>
        # 'progress' 변수 계산을 추가합니다.
        progress = (frame_count / total_frames) * 100

        # 진행률을 한 줄에 업데이트하여 깔끔하게 표시
        print(f"\r⏳ 진행률: {frame_count}/{total_frames} ({progress:.1f}%) | 프레임 처리 시간: {frame_time:.2f}초 | 예상 남은 시간(ETA): {eta_str}", end="")


    cap.release()
    out.release()

    total_time_str = time.strftime('%M분 %S초', time.gmtime(time.time() - start_time))
    print(f"\n\n✅ 처리 완료! 총 소요 시간: {total_time_str}")
    print(f"💾 결과가 '{output_video}' 파일로 저장되었습니다.")

    # --- Colab에 비디오 출력 ---
    try:
        mp4 = open(output_video,'rb').read()
        data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
        display(HTML(f"""
        <h3>결과 영상 미리보기: {output_video}</h3>
        <video width=800 controls style="border:1px solid #ccc;">
              <source src="{data_url}" type="video/mp4">
        </video>
        """))
    except Exception as e:
        print(f"❌ Colab에 비디오를 표시하는 중 오류 발생: {e}")
        print(f"'{output_video}' 파일을 직접 다운로드하여 확인해주세요.")


# --- 실행 코드 ---
# 처리할 영상 파일 경로를 지정해주세요.
INPUT_VIDEO_PATH = '/content/2.mp4'
OUTPUT_VIDEO_PATH = '/content/2_segmented_output.mp4'

process_video_all_objects(INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)